# LAB 2 – Bài 3: Phân loại món ăn VinaFood21 bằng ResNet-18

## Phiên bản bài giảng dành cho sinh viên mới học Deep Learning

Trong bài này, chúng ta xây dựng một hệ thống phân loại 21 món ăn Việt Nam bằng **ResNet-18**.

Notebook tập trung vào hai mục tiêu:

1. Hiểu ý tưởng **residual learning** và skip connection.
2. Thực hiện một quy trình thí nghiệm đúng, có thể tái lập và đánh giá công bằng.

## Mục tiêu học tập

Sau khi hoàn thành bài, sinh viên có thể:

1. Đọc dữ liệu ảnh theo cấu trúc thư mục bằng `ImageFolder`.
2. Tạo transform riêng cho train và validation/test.
3. Chia validation có phân tầng theo lớp.
4. Giải thích residual block và identity shortcut.
5. Theo dõi kích thước tensor qua ResNet-18.
6. Huấn luyện mô hình bằng mixed precision.
7. Lưu checkpoint dựa trên validation macro F1.
8. Đánh giá bằng accuracy, precision, recall, macro F1 và top-3 accuracy.
9. Phân tích confusion matrix và các ảnh dự đoán sai.

> **Nguyên tắc quan trọng:** Test set chỉ được sử dụng sau khi đã hoàn tất việc lựa chọn mô hình.

# 1. Cấu trúc dữ liệu

Dữ liệu VinaFood21 cần có cấu trúc:

```text
VinaFood21/
├── train/
│   ├── Banh_chung/
│   │   ├── image_001.jpg
│   │   └── ...
│   ├── Banh_mi/
│   └── ...
└── test/
    ├── Banh_chung/
    ├── Banh_mi/
    └── ...
```

Mỗi thư mục con tương ứng một lớp.

Trên Kaggle, đường dẫn thường là:

```text
/kaggle/input/vinafood21/VinaFood21
```

Trên máy cá nhân, có thể đặt thư mục `VinaFood21` cùng cấp với notebook.

# 2. Cài đặt thư viện

Chỉ chạy khi môi trường chưa có thư viện:

```python
%pip install torch torchvision scikit-learn matplotlib pillow
```

In [ ]:
# Bỏ dấu # ở dòng dưới nếu cần cài thư viện.
# %pip install torch torchvision scikit-learn matplotlib pillow

# 3. Import thư viện và thiết lập random seed

## Vì sao cần random seed?

Trong quá trình huấn luyện có nhiều yếu tố ngẫu nhiên:

- Khởi tạo trọng số.
- Thứ tự dữ liệu sau khi shuffle.
- Data augmentation.
- Một số phép toán trên GPU.

Seed giúp thí nghiệm dễ tái lập và so sánh hơn.

In [ ]:
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms


SEED = 42


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Ưu tiên tính tái lập hơn tốc độ.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Thiết bị:", device)
print("Phiên bản PyTorch:", torch.__version__)

# 4. Cấu hình thí nghiệm

Các siêu tham số được đặt chung để dễ thay đổi và ghi lại trong báo cáo.

## Lựa chọn ban đầu

- Kích thước ảnh: 224×224.
- Batch size: 32.
- Optimizer: AdamW.
- Learning rate: 0.001.
- Validation: 15% dữ liệu train.
- Checkpoint được chọn bằng validation macro F1.

In [ ]:
NUM_CLASSES = 21
INPUT_SIZE = 224
RESIZE_SIZE = 256

BATCH_SIZE = 32
MAX_EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VALIDATION_RATIO = 0.15
EARLY_STOPPING_PATIENCE = 4

NUM_WORKERS = 2 if os.name != "nt" else 0
USE_MIXED_PRECISION = device.type == "cuda"

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/resnet_output")
else:
    OUTPUT_DIR = Path("./resnet_output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "best_resnet18_vinafood21.pth"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"
TEST_RESULT_PATH = OUTPUT_DIR / "test_result.json"

print("Output directory:", OUTPUT_DIR.resolve())
print("Mixed precision:", USE_MIXED_PRECISION)

# 5. Tìm thư mục dữ liệu

Không nên viết cứng một đường dẫn chỉ đúng trên một máy.

Có thể sửa `MANUAL_DATA_ROOT` nếu dữ liệu nằm ở vị trí khác.

In [ ]:
# Ví dụ:
# MANUAL_DATA_ROOT = Path("/content/VinaFood21")
MANUAL_DATA_ROOT = None


def find_data_root(manual_root=None):
    candidates = []

    if manual_root is not None:
        candidates.append(Path(manual_root))

    candidates.extend(
        [
            Path("/kaggle/input/vinafood21/VinaFood21"),
            Path("/kaggle/input/vinafood21"),
            Path("./VinaFood21"),
            Path("./vinafood21/VinaFood21"),
            Path("./data/VinaFood21"),
            Path("./dataset/VinaFood21"),
            Path("../VinaFood21"),
        ]
    )

    for candidate in candidates:
        if (
            (candidate / "train").is_dir()
            and (candidate / "test").is_dir()
        ):
            return candidate

    checked_paths = "\n".join(
        f"- {path.resolve()}" for path in candidates
    )

    raise FileNotFoundError(
        "Không tìm thấy VinaFood21/train và VinaFood21/test.\n"
        "Các vị trí đã kiểm tra:\n"
        f"{checked_paths}\n\n"
        "Hãy sửa biến MANUAL_DATA_ROOT."
    )


DATA_ROOT = find_data_root(MANUAL_DATA_ROOT)
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

print("Data root:", DATA_ROOT.resolve())
print("Train directory:", TRAIN_DIR.resolve())
print("Test directory:", TEST_DIR.resolve())

# PHẦN A – XỬ LÝ DỮ LIỆU

# 6. Transform cho train và validation/test

## Train transform

Train transform có các phép biến đổi ngẫu nhiên để tạo nhiều phiên bản của ảnh:

- Random resized crop.
- Lật ngang.
- Xoay nhẹ.
- Thay đổi sáng, tương phản và độ bão hòa.

Đây là **data augmentation**, giúp giảm overfitting.

## Evaluation transform

Validation và test dùng transform xác định:

- Resize.
- Center crop.
- Chuyển thành Tensor.
- Normalize.

Không dùng augmentation ngẫu nhiên khi đánh giá vì kết quả cần ổn định.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

normalize = transforms.Normalize(
    mean=IMAGENET_MEAN,
    std=IMAGENET_STD,
)

train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(
            INPUT_SIZE,
            scale=(0.75, 1.0),
            ratio=(0.90, 1.10),
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.15,
        ),
        transforms.ToTensor(),
        normalize,
    ]
)

evaluation_transform = transforms.Compose(
    [
        transforms.Resize(RESIZE_SIZE),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.ToTensor(),
        normalize,
    ]
)

print(train_transform)

# 7. Vì sao dùng ImageNet mean và standard deviation?

Bộ giá trị:

```text
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
```

thường được dùng cho ảnh RGB đầu vào của các mô hình torchvision.

Lợi ích:

- Các kênh màu có quy mô ổn định.
- Tương thích khi mở rộng sang transfer learning.
- Không cần tự viết Dataset chỉ để đọc ảnh theo thư mục.

# 8. Đọc dữ liệu bằng `ImageFolder`

`ImageFolder` tự động:

- Tìm các thư mục lớp.
- Gán chỉ số cho từng lớp.
- Đọc đường dẫn ảnh.
- Trả về cặp `(image, label)`.

Vì mọi ảnh sau transform đều có cùng kích thước, `DataLoader` mặc định tự ghép batch. Không cần viết `collate_fn`.

In [ ]:
train_full_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=train_transform,
)

validation_full_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=evaluation_transform,
)

test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=evaluation_transform,
)

print("Số ảnh trong thư mục train:", len(train_full_dataset))
print("Số ảnh test:", len(test_dataset))
print("Số lớp:", len(train_full_dataset.classes))
print("Tên lớp:", train_full_dataset.classes)

# 9. Kiểm tra ánh xạ tên lớp

Train và test phải có cùng:

```text
tên lớp → chỉ số
```

Nếu không, mô hình có thể dự đoán đúng chỉ số nhưng tên món ăn bị ánh xạ sai.

In [ ]:
if train_full_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError(
        "class_to_idx của train và test không giống nhau. "
        "Hãy kiểm tra tên các thư mục lớp."
    )

if len(train_full_dataset.classes) != NUM_CLASSES:
    raise ValueError(
        f"NUM_CLASSES={NUM_CLASSES}, nhưng dữ liệu có "
        f"{len(train_full_dataset.classes)} lớp."
    )

class_names = train_full_dataset.classes
class_to_idx = train_full_dataset.class_to_idx

print("Ánh xạ lớp hợp lệ.")
print(class_to_idx)

# 10. Chia train và validation có phân tầng

## Train set

Dùng để cập nhật trọng số.

## Validation set

Dùng để:

- Chọn checkpoint.
- Điều chỉnh learning rate.
- Early stopping.
- So sánh các cấu hình.

## Test set

Chỉ dùng một lần ở cuối.

`stratify` giúp mỗi lớp có tỷ lệ gần giống nhau trong train và validation.

In [ ]:
all_indices = np.arange(len(train_full_dataset))
all_targets = np.asarray(train_full_dataset.targets)

class_counts_before_split = np.bincount(
    all_targets,
    minlength=NUM_CLASSES,
)

if class_counts_before_split.min() < 2:
    raise ValueError(
        "Mỗi lớp cần ít nhất 2 ảnh để chia có phân tầng."
    )

train_indices, validation_indices = train_test_split(
    all_indices,
    test_size=VALIDATION_RATIO,
    random_state=SEED,
    shuffle=True,
    stratify=all_targets,
)

train_dataset = Subset(
    train_full_dataset,
    train_indices,
)

validation_dataset = Subset(
    validation_full_dataset,
    validation_indices,
)

print("Số ảnh train:", len(train_dataset))
print("Số ảnh validation:", len(validation_dataset))
print("Số ảnh test:", len(test_dataset))

# 11. Khảo sát phân bố lớp

Accuracy có thể gây hiểu nhầm nếu dữ liệu mất cân bằng.

Ví dụ, một mô hình dự đoán tốt lớp có nhiều ảnh nhưng kém ở lớp ít ảnh vẫn có accuracy khá cao.

Vì vậy bài này sử dụng thêm macro F1.

In [ ]:
train_targets = all_targets[train_indices]
validation_targets = all_targets[validation_indices]

train_counts = np.bincount(
    train_targets,
    minlength=NUM_CLASSES,
)

validation_counts = np.bincount(
    validation_targets,
    minlength=NUM_CLASSES,
)

for class_index, class_name in enumerate(class_names):
    print(
        f"{class_index:02d} {class_name:25s} | "
        f"train: {train_counts[class_index]:4d} | "
        f"validation: {validation_counts[class_index]:4d}"
    )

In [ ]:
positions = np.arange(NUM_CLASSES)

plt.figure(figsize=(14, 5))
plt.bar(positions, train_counts)
plt.xticks(positions, class_names, rotation=90)
plt.xlabel("Lớp")
plt.ylabel("Số ảnh")
plt.title("Phân bố lớp trong train set")
plt.tight_layout()
plt.show()

# 12. Tạo DataLoader

- Train dùng `shuffle=True`.
- Validation và test dùng `shuffle=False`.
- `pin_memory=True` hỗ trợ truyền dữ liệu sang GPU.
- `worker_init_fn` hỗ trợ tính tái lập khi có nhiều worker.

In [ ]:
loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

pin_memory = device.type == "cuda"

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker,
    generator=loader_generator,
)

validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=NUM_WORKERS > 0,
    worker_init_fn=seed_worker,
)

first_images, first_labels = next(iter(train_loader))

print("Batch ảnh:", first_images.shape)
print("Batch nhãn:", first_labels.shape)
print("Kiểu dữ liệu:", first_images.dtype)

# 13. Hiển thị ảnh sau khi bỏ Normalize

Sau Normalize, ảnh không còn ở khoảng `[0, 1]`.

Muốn hiển thị đúng màu cần thực hiện phép biến đổi ngược.

In [ ]:
def denormalize_image(image_tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

    image = image_tensor.cpu() * std + mean
    return image.clamp(0, 1)


number_of_samples = min(8, len(first_images))

figure, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for index in range(number_of_samples):
    image = denormalize_image(first_images[index])
    image = image.permute(1, 2, 0)

    axes[index].imshow(image)
    axes[index].set_title(
        class_names[first_labels[index].item()]
    )
    axes[index].axis("off")

for index in range(number_of_samples, len(axes)):
    axes[index].axis("off")

plt.tight_layout()
plt.show()

# PHẦN B – LÝ THUYẾT RESNET

# 14. Vấn đề khi mạng ngày càng sâu

Trực giác ban đầu: thêm nhiều lớp sẽ giúp mô hình tốt hơn.

Tuy nhiên, mạng quá sâu có thể gặp:

- Gradient biến mất hoặc bùng nổ.
- Khó tối ưu.
- Training error tăng dù mô hình sâu có khả năng biểu diễn lớn hơn.
- Các lớp sâu khó học được phép biến đổi hữu ích.

ResNet đưa ra residual connection để giúp tín hiệu và gradient đi qua mạng dễ hơn.

# 15. Residual learning

Thay vì bắt một khối học trực tiếp ánh xạ:

\[
H(x)
\]

ResNet cho khối học phần dư:

\[
F(x) = H(x) - x
\]

Khi đó:

\[
H(x) = F(x) + x
\]

Biểu diễn trong code:

```python
residual = x
out = convolution_layers(x)
out = out + residual
out = relu(out)
```

Nếu phép biến đổi tối ưu gần identity, khối chỉ cần học `F(x)` gần 0.

# 16. Identity shortcut và projection shortcut

## Identity shortcut

Dùng khi đầu vào và đầu ra có cùng kích thước:

```text
F(x) + x
```

Không có thêm tham số.

## Projection shortcut

Dùng khi:

- Số channel thay đổi.
- Kích thước không gian giảm do stride 2.

Khi đó nhánh shortcut dùng convolution 1×1 để đưa `x` về cùng shape với `F(x)`.

# 17. BasicBlock của ResNet-18

ResNet-18 sử dụng BasicBlock:

```text
Input
  ├──────────────────────────────┐
  │                              │
  └→ Conv 3×3 → BN → ReLU        │
                 → Conv 3×3 → BN │
                                  ├→ Cộng → ReLU
  Shortcut ──────────────────────┘
```

Mỗi block có hai convolution 3×3.

ResNet-50 trở lên thường dùng Bottleneck block gồm convolution 1×1, 3×3 và 1×1.

# 18. Cấu trúc ResNet-18

```text
Input 3×224×224
→ Conv 7×7, stride 2
→ MaxPool 3×3, stride 2
→ Layer 1: 2 BasicBlocks, 64 channels
→ Layer 2: 2 BasicBlocks, 128 channels
→ Layer 3: 2 BasicBlocks, 256 channels
→ Layer 4: 2 BasicBlocks, 512 channels
→ Global Average Pooling
→ Fully Connected
→ 21 logits
```

Tên ResNet-18 xuất phát từ tổng số lớp có trọng số chính trong kiến trúc.

# 19. Vì sao giữ nguyên MaxPool mặc định?

ResNet-18 chuẩn trong torchvision sử dụng:

```python
MaxPool2d(kernel_size=3, stride=2, padding=1)
```

Padding 1 giúp kích thước không gian giảm theo thiết kế chuẩn.

Không nên tự đổi thành `padding=0` nếu đề bài không yêu cầu, vì:

- Làm thay đổi kích thước feature map.
- Không còn đúng kiến trúc ResNet-18 chuẩn.
- Gây khó khi so sánh với pretrained weights hoặc tài liệu.

# PHẦN C – XÂY DỰNG MÔ HÌNH

# 20. Tạo ResNet-18

Bài cơ sở huấn luyện từ đầu sử dụng:

```python
weights=None
```

`num_classes=21` làm lớp fully connected cuối cùng trả về 21 logits.

Không thêm Softmax vào `forward()` vì loss sẽ là CrossEntropyLoss.

In [ ]:
def create_resnet18(num_classes):
    return models.resnet18(
        weights=None,
        num_classes=num_classes,
    )


model = create_resnet18(NUM_CLASSES).to(device)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(model)
print(f"\nSố tham số có thể học: {trainable_parameters:,}")

# 21. Kiểm tra đầu ra trước khi huấn luyện

Với batch ảnh:

```text
(batch_size, 3, 224, 224)
```

đầu ra phải là:

```text
(batch_size, 21)
```

In [ ]:
model.eval()

dummy_images = torch.randn(
    2,
    3,
    INPUT_SIZE,
    INPUT_SIZE,
    device=device,
)

with torch.inference_mode():
    dummy_logits = model(dummy_images)

print("Input shape:", dummy_images.shape)
print("Output shape:", dummy_logits.shape)

assert dummy_logits.shape == (2, NUM_CLASSES)

del dummy_images, dummy_logits

if device.type == "cuda":
    torch.cuda.empty_cache()

# 22. Theo dõi kích thước tensor qua các stage

Đây là bước sanity check giúp sinh viên hiểu ResNet giảm kích thước ảnh như thế nào.

In [ ]:
def trace_resnet_shapes(model, input_size=224):
    model.eval()

    x = torch.zeros(
        1,
        3,
        input_size,
        input_size,
        device=device,
    )

    stages = [
        ("conv1", model.conv1),
        ("bn1", model.bn1),
        ("relu", model.relu),
        ("maxpool", model.maxpool),
        ("layer1", model.layer1),
        ("layer2", model.layer2),
        ("layer3", model.layer3),
        ("layer4", model.layer4),
        ("avgpool", model.avgpool),
    ]

    print("input".ljust(12), tuple(x.shape))

    with torch.inference_mode():
        for name, layer in stages:
            x = layer(x)
            print(name.ljust(12), tuple(x.shape))

        x = torch.flatten(x, 1)
        logits = model.fc(x)

    print("flatten".ljust(12), tuple(x.shape))
    print("logits".ljust(12), tuple(logits.shape))


trace_resnet_shapes(model, INPUT_SIZE)

# 23. Quan sát projection shortcut

Block đầu tiên của `layer2`, `layer3` và `layer4` phải:

- Giảm kích thước không gian.
- Tăng số channel.

Vì vậy chúng có nhánh `downsample` dùng convolution 1×1.

In [ ]:
print("Layer1 block0 downsample:")
print(model.layer1[0].downsample)

print("\nLayer2 block0 downsample:")
print(model.layer2[0].downsample)

print("\nLayer3 block0 downsample:")
print(model.layer3[0].downsample)

print("\nLayer4 block0 downsample:")
print(model.layer4[0].downsample)

# PHẦN D – LOSS, OPTIMIZER VÀ METRICS

# 24. CrossEntropyLoss

Mô hình trả về logits.

`CrossEntropyLoss` đã thực hiện phép tính tương đương LogSoftmax bên trong.

Không viết:

```python
logits = torch.softmax(logits, dim=1)
loss = loss_fn(logits, labels)
```

Softmax chỉ dùng khi cần hiển thị xác suất dự đoán.

# 25. AdamW và weight decay

AdamW được chọn vì:

- Dễ dùng cho người mới.
- Điều chỉnh mức cập nhật cho từng tham số.
- Tách weight decay khỏi cơ chế cập nhật của Adam.

Weight decay giúp hạn chế trọng số quá lớn và có thể giảm overfitting.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
)


def create_grad_scaler():
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=USE_MIXED_PRECISION,
        )
    except TypeError:
        return torch.cuda.amp.GradScaler(
            enabled=USE_MIXED_PRECISION,
        )


scaler = create_grad_scaler()

print(loss_fn)
print(optimizer)

# 26. Mixed precision

Khi dùng GPU, mixed precision có thể:

- Giảm bộ nhớ.
- Tăng tốc huấn luyện.
- Cho phép batch lớn hơn.

`GradScaler` giúp giảm nguy cơ gradient bị quá nhỏ.

Trên CPU, notebook tự động tắt mixed precision.

# 27. Các chỉ số đánh giá

## Accuracy

Tỷ lệ dự đoán đúng trên toàn bộ dữ liệu.

## Macro precision, recall và F1

Tính riêng cho từng lớp rồi lấy trung bình. Mỗi lớp có vai trò ngang nhau.

## Top-3 accuracy

Dự đoán được xem là đúng nếu nhãn thật nằm trong ba lớp có logits cao nhất.

Chỉ số này hữu ích khi nhiều món ăn có hình thức gần nhau.

In [ ]:
def evaluate_model(
    model,
    dataloader,
    loss_fn,
    return_predictions=False,
):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    total_top3_correct = 0

    all_labels = []
    all_predictions = []

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(
                device,
                non_blocking=True,
            )
            labels = labels.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type=device.type,
                enabled=USE_MIXED_PRECISION,
            ):
                logits = model(images)
                loss = loss_fn(logits, labels)

            predictions = logits.argmax(dim=1)
            top3_indices = logits.topk(
                k=min(3, NUM_CLASSES),
                dim=1,
            ).indices

            batch_size = images.size(0)

            total_loss += loss.item() * batch_size
            total_samples += batch_size

            total_top3_correct += (
                top3_indices
                .eq(labels.unsqueeze(1))
                .any(dim=1)
                .sum()
                .item()
            )

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(
                predictions.cpu().numpy()
            )

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_predictions)

    metrics = {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "top3_accuracy": (
            total_top3_correct / total_samples
        ),
    }

    if return_predictions:
        metrics["y_true"] = y_true
        metrics["y_pred"] = y_pred

    return metrics

# PHẦN E – HUẤN LUYỆN

# 28. Hàm huấn luyện một epoch

Trong mỗi batch:

1. Chuyển dữ liệu sang thiết bị.
2. Xóa gradient cũ.
3. Forward.
4. Tính loss.
5. Backward.
6. Optimizer cập nhật trọng số.

Loss được tính trung bình theo số mẫu, không chỉ trung bình đều giữa các batch.

In [ ]:
def train_one_epoch(
    model,
    dataloader,
    loss_fn,
    optimizer,
    scaler,
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in dataloader:
        images = images.to(
            device,
            non_blocking=True,
        )
        labels = labels.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=device.type,
            enabled=USE_MIXED_PRECISION,
        ):
            logits = model(images)
            loss = loss_fn(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        predictions = logits.argmax(dim=1)
        batch_size = images.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (
            predictions == labels
        ).sum().item()
        total_samples += batch_size

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
    }

# 29. Checkpoint, scheduler và early stopping

Sau mỗi epoch:

1. Huấn luyện trên train.
2. Đánh giá validation.
3. Scheduler theo dõi validation loss.
4. Lưu checkpoint nếu validation macro F1 tốt hơn.
5. Dừng sớm nếu nhiều epoch không cải thiện.

Test set không xuất hiện trong vòng lặp.

In [ ]:
history = {
    "train_loss": [],
    "train_accuracy": [],
    "validation_loss": [],
    "validation_accuracy": [],
    "validation_f1_macro": [],
    "validation_top3_accuracy": [],
    "learning_rate": [],
}

best_validation_f1 = float("-inf")
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_metrics = train_one_epoch(
        model=model,
        dataloader=train_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        scaler=scaler,
    )

    validation_metrics = evaluate_model(
        model=model,
        dataloader=validation_loader,
        loss_fn=loss_fn,
    )

    scheduler.step(validation_metrics["loss"])

    current_learning_rate = (
        optimizer.param_groups[0]["lr"]
    )

    history["train_loss"].append(
        float(train_metrics["loss"])
    )
    history["train_accuracy"].append(
        float(train_metrics["accuracy"])
    )
    history["validation_loss"].append(
        float(validation_metrics["loss"])
    )
    history["validation_accuracy"].append(
        float(validation_metrics["accuracy"])
    )
    history["validation_f1_macro"].append(
        float(validation_metrics["f1_macro"])
    )
    history["validation_top3_accuracy"].append(
        float(validation_metrics["top3_accuracy"])
    )
    history["learning_rate"].append(
        float(current_learning_rate)
    )

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"Train loss: {train_metrics['loss']:.4f} | "
        f"Train acc: {train_metrics['accuracy']:.4f} | "
        f"Val loss: {validation_metrics['loss']:.4f} | "
        f"Val acc: {validation_metrics['accuracy']:.4f} | "
        f"Val macro F1: {validation_metrics['f1_macro']:.4f} | "
        f"Val top-3: {validation_metrics['top3_accuracy']:.4f} | "
        f"LR: {current_learning_rate:.2e}"
    )

    if (
        validation_metrics["f1_macro"]
        > best_validation_f1
    ):
        best_validation_f1 = (
            validation_metrics["f1_macro"]
        )
        epochs_without_improvement = 0

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": (
                optimizer.state_dict()
            ),
            "scheduler_state_dict": (
                scheduler.state_dict()
            ),
            "validation_metrics": {
                key: float(value)
                for key, value
                in validation_metrics.items()
            },
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "config": {
                "seed": SEED,
                "num_classes": NUM_CLASSES,
                "input_size": INPUT_SIZE,
                "batch_size": BATCH_SIZE,
                "max_epochs": MAX_EPOCHS,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "validation_ratio": VALIDATION_RATIO,
            },
        }

        torch.save(
            checkpoint,
            CHECKPOINT_PATH,
        )

        print(
            "  -> Đã lưu checkpoint mới. "
            f"Validation macro F1 = "
            f"{best_validation_f1:.4f}"
        )
    else:
        epochs_without_improvement += 1

        print(
            "  -> Chưa cải thiện. "
            f"Early stopping: "
            f"{epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print(
            "\nDừng sớm vì validation macro F1 "
            "không còn cải thiện."
        )
        break

with HISTORY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        history,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("\nĐã lưu history:", HISTORY_PATH.resolve())
print("Checkpoint tốt nhất:", CHECKPOINT_PATH.resolve())

# 30. Biểu đồ loss

Dấu hiệu overfitting thường gặp:

- Train loss tiếp tục giảm.
- Validation loss bắt đầu tăng.
- Validation F1 không còn cải thiện.

In [ ]:
epochs_ran = range(
    1,
    len(history["train_loss"]) + 1,
)

plt.figure(figsize=(9, 5))
plt.plot(
    epochs_ran,
    history["train_loss"],
    marker="o",
    label="Train loss",
)
plt.plot(
    epochs_ran,
    history["validation_loss"],
    marker="o",
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train loss và validation loss")
plt.grid(True)
plt.legend()
plt.show()

# 31. Biểu đồ accuracy và macro F1

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    epochs_ran,
    history["train_accuracy"],
    marker="o",
    label="Train accuracy",
)
plt.plot(
    epochs_ran,
    history["validation_accuracy"],
    marker="o",
    label="Validation accuracy",
)
plt.plot(
    epochs_ran,
    history["validation_f1_macro"],
    marker="o",
    label="Validation macro F1",
)
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Accuracy và macro F1")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.show()

# 32. Biểu đồ top-3 accuracy

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    epochs_ran,
    history["validation_top3_accuracy"],
    marker="o",
    label="Validation top-3 accuracy",
)
plt.xlabel("Epoch")
plt.ylabel("Top-3 accuracy")
plt.title("Validation top-3 accuracy")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.show()

# 33. Biểu đồ learning rate

Scheduler giảm learning rate khi validation loss không cải thiện.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    epochs_ran,
    history["learning_rate"],
    marker="o",
)
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.title("Learning rate theo epoch")
plt.grid(True)
plt.show()

# PHẦN F – ĐÁNH GIÁ CUỐI CÙNG

# 34. Tải lại checkpoint tốt nhất

Epoch cuối chưa chắc là epoch tốt nhất.

Phải tải lại mô hình có validation macro F1 cao nhất trước khi test.

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print("Checkpoint epoch:", checkpoint["epoch"])
print(
    "Validation macro F1 tốt nhất:",
    f"{checkpoint['validation_metrics']['f1_macro']:.4f}",
)

# 35. Chỉ bây giờ mới dùng test set

Test score là kết quả cuối cùng.

Không dùng kết quả test để:

- Chọn lại epoch.
- Đổi learning rate.
- Thay augmentation.
- Chọn kiến trúc khác.

Các quyết định đó phải dựa trên validation set.

In [ ]:
test_metrics = evaluate_model(
    model=model,
    dataloader=test_loader,
    loss_fn=loss_fn,
    return_predictions=True,
)

print("KẾT QUẢ CUỐI CÙNG TRÊN TEST SET")
print("-" * 50)
print(f"Loss:            {test_metrics['loss']:.4f}")
print(f"Accuracy:        {test_metrics['accuracy']:.4f}")
print(f"Macro precision: {test_metrics['precision_macro']:.4f}")
print(f"Macro recall:    {test_metrics['recall_macro']:.4f}")
print(f"Macro F1:        {test_metrics['f1_macro']:.4f}")
print(f"Top-3 accuracy:  {test_metrics['top3_accuracy']:.4f}")

In [ ]:
serializable_test_result = {
    key: float(value)
    for key, value in test_metrics.items()
    if key not in {"y_true", "y_pred"}
}

with TEST_RESULT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        serializable_test_result,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Đã lưu kết quả:", TEST_RESULT_PATH.resolve())

# 36. Classification report

Báo cáo theo từng lớp giúp phát hiện:

- Lớp precision thấp.
- Lớp recall thấp.
- Lớp F1 thấp nhất.
- Chênh lệch giữa các món ăn.

In [ ]:
print(
    classification_report(
        test_metrics["y_true"],
        test_metrics["y_pred"],
        target_names=class_names,
        digits=4,
        zero_division=0,
    )
)

# 37. Confusion matrix chuẩn hóa

- Hàng: nhãn thật.
- Cột: nhãn dự đoán.
- Mỗi hàng được chuẩn hóa về tỷ lệ.

Các ô lớn ngoài đường chéo là những cặp món ăn thường bị nhầm.

In [ ]:
normalized_matrix = confusion_matrix(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    normalize="true",
)

display = ConfusionMatrixDisplay(
    confusion_matrix=normalized_matrix,
    display_labels=class_names,
)

figure, axis = plt.subplots(figsize=(16, 16))
display.plot(
    ax=axis,
    values_format=".2f",
    xticks_rotation=90,
)
axis.set_title("Normalized confusion matrix")
plt.tight_layout()
plt.show()

# 38. Accuracy theo từng lớp

Đường chéo của confusion matrix chuẩn hóa tương ứng recall của từng lớp.

In [ ]:
per_class_accuracy = np.diag(normalized_matrix)
positions = np.arange(NUM_CLASSES)

plt.figure(figsize=(14, 5))
plt.bar(positions, per_class_accuracy)
plt.xticks(positions, class_names, rotation=90)
plt.ylim(0, 1)
plt.xlabel("Lớp")
plt.ylabel("Accuracy theo lớp")
plt.title("Độ chính xác của từng lớp")
plt.tight_layout()
plt.show()

hardest_class_index = int(
    np.argmin(per_class_accuracy)
)

print(
    "Lớp khó nhất:",
    class_names[hardest_class_index],
)
print(
    "Accuracy:",
    f"{per_class_accuracy[hardest_class_index]:.4f}",
)

# 39. Các cặp lớp bị nhầm nhiều nhất

In [ ]:
raw_matrix = confusion_matrix(
    test_metrics["y_true"],
    test_metrics["y_pred"],
)

confusion_without_diagonal = raw_matrix.copy()
np.fill_diagonal(confusion_without_diagonal, 0)

flat_indices = np.argsort(
    confusion_without_diagonal.ravel()
)[::-1]

print("Các cặp lớp bị nhầm nhiều nhất:")

number_printed = 0

for flat_index in flat_indices:
    true_index, predicted_index = np.unravel_index(
        flat_index,
        confusion_without_diagonal.shape,
    )

    count = confusion_without_diagonal[
        true_index,
        predicted_index,
    ]

    if count == 0:
        break

    print(
        f"- Thật: {class_names[true_index]:25s} "
        f"→ Đoán: {class_names[predicted_index]:25s} "
        f"| {count} ảnh"
    )

    number_printed += 1

    if number_printed >= 10:
        break

# 40. Hiển thị ảnh dự đoán sai

Quan sát ảnh sai giúp tìm nguyên nhân:

- Hai món có hình thức tương tự.
- Ảnh quá tối hoặc mờ.
- Món ăn chiếm diện tích nhỏ.
- Nền ảnh gây nhiễu.
- Nhãn dữ liệu có thể sai.

In [ ]:
def collect_misclassified_samples(
    model,
    dataloader,
    max_samples=12,
):
    model.eval()
    samples = []

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            with torch.autocast(
                device_type=device.type,
                enabled=USE_MIXED_PRECISION,
            ):
                logits = model(images)

            probabilities = torch.softmax(
                logits,
                dim=1,
            )
            predictions = logits.argmax(dim=1)

            wrong_indices = (
                predictions != labels
            ).nonzero(as_tuple=True)[0]

            for index in wrong_indices:
                predicted_index = (
                    predictions[index].item()
                )

                samples.append(
                    {
                        "image": images[index].cpu(),
                        "true_index": labels[index].item(),
                        "predicted_index": predicted_index,
                        "confidence": probabilities[
                            index,
                            predicted_index,
                        ].item(),
                    }
                )

                if len(samples) >= max_samples:
                    return samples

    return samples


misclassified_samples = collect_misclassified_samples(
    model=model,
    dataloader=test_loader,
    max_samples=12,
)

print(
    "Số ảnh sai được lấy:",
    len(misclassified_samples),
)

In [ ]:
if misclassified_samples:
    figure, axes = plt.subplots(
        3,
        4,
        figsize=(15, 11),
    )
    axes = axes.flatten()

    for axis, sample in zip(
        axes,
        misclassified_samples,
    ):
        image = denormalize_image(
            sample["image"]
        )
        image = image.permute(1, 2, 0)

        true_name = class_names[
            sample["true_index"]
        ]
        predicted_name = class_names[
            sample["predicted_index"]
        ]

        axis.imshow(image)
        axis.set_title(
            f"Thật: {true_name}\n"
            f"Đoán: {predicted_name}\n"
            f"Tự tin: {sample['confidence']:.2%}",
            fontsize=9,
        )
        axis.axis("off")

    for axis in axes[len(misclassified_samples):]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy ảnh dự đoán sai.")

# 41. Dự đoán một ảnh bên ngoài

Softmax chỉ được dùng ở đây để chuyển logits thành xác suất dễ diễn giải.

Hàm trả về ba lớp có xác suất cao nhất.

In [ ]:
def predict_image(
    model,
    image_path,
    transform,
    class_names,
    top_k=3,
):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    image_tensor = (
        transform(image)
        .unsqueeze(0)
        .to(device)
    )

    with torch.inference_mode():
        with torch.autocast(
            device_type=device.type,
            enabled=USE_MIXED_PRECISION,
        ):
            logits = model(image_tensor)

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        top_probabilities, top_indices = (
            probabilities.topk(
                k=min(top_k, len(class_names)),
                dim=1,
            )
        )

    results = []

    for probability, class_index in zip(
        top_probabilities[0].cpu(),
        top_indices[0].cpu(),
    ):
        results.append(
            {
                "class_name": (
                    class_names[class_index.item()]
                ),
                "probability": probability.item(),
            }
        )

    return image, results


# Ví dụ:
# image_path = TEST_DIR / class_names[0] / "ten_anh.jpg"
#
# image, results = predict_image(
#     model=model,
#     image_path=image_path,
#     transform=evaluation_transform,
#     class_names=class_names,
# )
#
# plt.figure(figsize=(6, 6))
# plt.imshow(image)
# plt.axis("off")
# plt.show()
#
# for result in results:
#     print(
#         result["class_name"],
#         f"{result['probability']:.2%}",
#     )

# PHẦN G – MỞ RỘNG: TRANSFER LEARNING

# 42. Khi nào nên dùng pretrained weights?

Huấn luyện ResNet-18 từ đầu cần đủ dữ liệu và thời gian.

Với tập dữ liệu nhỏ, transfer learning thường:

- Hội tụ nhanh hơn.
- Cho validation score tốt hơn.
- Cần ít epoch hơn.

Đây là một thí nghiệm riêng và vẫn phải được chọn bằng validation set.

In [ ]:
# MÃ THAM KHẢO – KHÔNG CHẠY CHUNG VỚI THÍ NGHIỆM TỪ ĐẦU
#
# from torchvision.models import ResNet18_Weights
#
# pretrained_model = models.resnet18(
#     weights=ResNet18_Weights.DEFAULT,
# )
#
# pretrained_model.fc = nn.Linear(
#     pretrained_model.fc.in_features,
#     NUM_CLASSES,
# )
#
# pretrained_model = pretrained_model.to(device)
#
# Có thể đóng băng backbone trong một số epoch đầu:
#
# for parameter in pretrained_model.parameters():
#     parameter.requires_grad = False
#
# for parameter in pretrained_model.fc.parameters():
#     parameter.requires_grad = True

# PHẦN H – TỔNG KẾT

# 43. Quy trình chuẩn

```text
1. Kiểm tra cấu trúc dữ liệu
2. Kiểm tra class_to_idx
3. Tạo train và evaluation transform
4. Chia train–validation có phân tầng
5. Khảo sát phân bố lớp
6. Tạo DataLoader
7. Xây dựng ResNet-18 chuẩn
8. Kiểm tra output shape
9. Theo dõi tensor qua từng stage
10. Huấn luyện trên train
11. Đánh giá validation
12. Lưu checkpoint tốt nhất
13. Early stopping
14. Tải lại checkpoint
15. Test đúng một lần
16. Phân tích confusion matrix và ảnh sai
```

# 44. Các lỗi thường gặp

## Lỗi 1: Dùng test set sau mỗi epoch

Test set bị rò rỉ vào quá trình lựa chọn mô hình.

## Lỗi 2: Dùng augmentation cho validation

Kết quả validation thay đổi ngẫu nhiên.

## Lỗi 3: Tự viết Dataset khi `ImageFolder` đã đáp ứng

Code dài hơn, dễ lỗi và phải tự xử lý phần mở rộng file.

## Lỗi 4: Tự thay MaxPool chuẩn thành padding 0

Làm thay đổi kiến trúc ResNet-18 mà không có lý do rõ ràng.

## Lỗi 5: Thêm Softmax trước CrossEntropyLoss

Không cần thiết.

## Lỗi 6: Chỉ lưu `state_dict` nhưng không ghi cấu hình

Khó tái tạo thí nghiệm sau này.

## Lỗi 7: Không tải lại checkpoint tốt nhất

Epoch cuối chưa chắc tốt nhất.

## Lỗi 8: Chỉ báo accuracy

Có thể che giấu lớp có chất lượng thấp.

## Lỗi 9: Nộp notebook chưa chạy

Không thể kiểm tra kết quả thực tế.

# 45. Câu hỏi tự kiểm tra

1. Residual block học `F(x)` thay vì học trực tiếp điều gì?
2. Vì sao phải cộng shortcut với nhánh convolution?
3. Khi nào cần projection shortcut?
4. ResNet-18 sử dụng BasicBlock hay Bottleneck?
5. Vì sao không nên đổi MaxPool mặc định tùy ý?
6. Validation set được dùng để làm gì?
7. Macro F1 khác accuracy ở điểm nào?
8. Vì sao cần tải lại checkpoint tốt nhất?
9. Dấu hiệu nào cho thấy overfitting?
10. Transfer learning phù hợp trong trường hợp nào?

# 46. Bài tập mở rộng

## Bài 1 – So sánh optimizer

So sánh AdamW và SGD momentum bằng validation macro F1.

## Bài 2 – So sánh augmentation

Thử ba mức augmentation: không augmentation, nhẹ và mạnh.

## Bài 3 – Transfer learning

So sánh huấn luyện từ đầu và pretrained ImageNet.

## Bài 4 – ResNet-18 và ResNet-34

So sánh số tham số, thời gian huấn luyện và validation score.

## Bài 5 – Xử lý mất cân bằng

Thử class weights hoặc `WeightedRandomSampler`.

## Bài 6 – Phân tích residual feature

Dùng forward hook để quan sát feature map trước và sau một BasicBlock.

## Bài 7 – Phân tích lỗi

Chọn ba cặp món ăn bị nhầm nhiều nhất và giải thích bằng ảnh.

## Bài 8 – Báo cáo

Báo cáo gồm:

1. Cấu trúc dữ liệu.
2. Kiến trúc ResNet-18.
3. Cấu hình thí nghiệm.
4. Biểu đồ huấn luyện.
5. Validation tốt nhất.
6. Test cuối cùng.
7. Confusion matrix.
8. Ảnh sai.
9. Hướng cải tiến.

# Checklist trước khi nộp

- [ ] Đã chạy `Restart & Run All`.
- [ ] Không có ô code báo lỗi.
- [ ] Có đúng 21 lớp.
- [ ] Train và test có cùng `class_to_idx`.
- [ ] Validation không dùng augmentation ngẫu nhiên.
- [ ] Không dùng test để chọn mô hình.
- [ ] Không tự thay đổi stem chuẩn của ResNet nếu đề không yêu cầu.
- [ ] Có output từng epoch.
- [ ] Có biểu đồ loss và metrics.
- [ ] Có checkpoint tốt nhất.
- [ ] Có test result cuối cùng.
- [ ] Có classification report.
- [ ] Có confusion matrix.
- [ ] Có phân tích ảnh dự đoán sai.
- [ ] Đã lưu notebook sau khi chạy.